In [1]:
from datasets import load_dataset
from random import random
import pandas as pd

/home/hein/hof217/.conda/envs/sd2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

ds = load_dataset("eugene6923/Self-Bench")
ds = ds["train"]

In [3]:
# unique "class" in self bench dataset
unique_classes = set(ds["class"])
print(len(unique_classes))
print(unique_classes)

80
{'microwave', 'airplane', 'fire hydrant', 'vase', 'sandwich', 'cake', 'bed', 'cup', 'spoon', 'tv', 'knife', 'potted plant', 'computer mouse', 'toilet', 'banana', 'baseball bat', 'baseball glove', 'bowl', 'motorcycle', 'bench', 'scissors', 'toaster', 'traffic light', 'tie', 'parking meter', 'clock', 'giraffe', 'sports ball', 'sheep', 'cell phone', 'horse', 'cat', 'couch', 'tennis racket', 'bird', 'teddy bear', 'computer keyboard', 'orange', 'toothbrush', 'surfboard', 'boat', 'dining table', 'bear', 'apple', 'backpack', 'dog', 'car', 'bottle', 'handbag', 'bus', 'book', 'skis', 'broccoli', 'kite', 'oven', 'frisbee', 'chair', 'carrot', 'snowboard', 'sink', 'pizza', 'bicycle', 'suitcase', 'hair drier', 'truck', 'wine glass', 'hot dog', 'person', 'skateboard', 'fork', 'tv remote', 'umbrella', 'zebra', 'elephant', 'train', 'stop sign', 'refrigerator', 'laptop', 'donut', 'cow'}


In [4]:
# read the unique class pairs from the csv file: class_pairs_from_143851.csv
class_pairs = pd.read_csv("class_pairs_from_143851.csv", header=None).values
class_pairs = [(row[0], row[1]) for row in class_pairs]
print(len(class_pairs))
print(class_pairs)

101
[('class1', 'class2'), ('giraffe', 'hair drier'), ('broccoli', 'skis'), ('book', 'sports ball'), ('bench', 'cow'), ('computer keyboard', 'skateboard'), ('toilet', 'potted plant'), ('oven', 'vase'), ('bed', 'book'), ('cell phone', 'bear'), ('sports ball', 'refrigerator'), ('baseball bat', 'dog'), ('orange', 'traffic light'), ('teddy bear', 'orange'), ('skis', 'tennis racket'), ('zebra', 'bicycle'), ('sheep', 'computer mouse'), ('cell phone', 'orange'), ('knife', 'umbrella'), ('motorcycle', 'carrot'), ('traffic light', 'horse'), ('surfboard', 'tv'), ('carrot', 'truck'), ('giraffe', 'fire hydrant'), ('elephant', 'toaster'), ('bowl', 'dog'), ('bench', 'motorcycle'), ('frisbee', 'handbag'), ('skis', 'cat'), ('surfboard', 'spoon'), ('bicycle', 'bus'), ('car', 'laptop'), ('frisbee', 'cat'), ('skis', 'motorcycle'), ('toilet', 'cat'), ('snowboard', 'boat'), ('broccoli', 'bird'), ('scissors', 'airplane'), ('bowl', 'banana'), ('baseball glove', 'bicycle'), ('microwave', 'skis'), ('banana', 's

In [8]:
all_simple_positional = ["to the left of", "to the right of", "above", "behind", "on", "in front of",
                         "on top of", "below", "under", "over",]
 
misc_positional = ["between", "inside", "outside"] 

words = ["holding", "touching", "not holding", "not touching"]

negative_words = ["and not", "without"]


In [9]:
# # make groups of 3 entries each
# simple_positional_word_groups = [
#     all_simple_positional[i:i + 3] for i in range(0, len(all_simple_positional), 3)
# ]
# print(len(simple_positional_word_groups))
# print(simple_positional_word_groups)

In [10]:
def generate_prompts(category_list, category="simple_positional"):
    """
    generate 100 prompts each of the form 
    "a photo of a {class} {simple_positional_word} a {another_class}"
    """

    # prompts_data contains the generated prompt, the classes used, category (positional_word, verb, etc) and the category phrase used
    prompts_data = []
    
    # take the unqiue class pairs generated earlier
    sampled_pairs = class_pairs
    
    # Generate prompts for each phrase using the same 100 class pairs
    for class1, class2 in sampled_pairs:
        for phrase in category_list:
            if phrase == "between":
                prompt = f"a photo of a {class1} {phrase} {class2}s"
            else:
                prompt = f"a photo of a {class1} {phrase} a {class2}"
            prompts_data.append((prompt, class1, class2, category, phrase))
    
    print(f"Generated {len(prompts_data)} prompts")
    print(prompts_data[:5])

    # save prompts_data to a csv file with unique identifier
    hash_suffix = abs(hash(tuple(category_list))) % (10 ** 8)
    csv_name = f"simple_positional_{hash_suffix}.csv"
    df = pd.DataFrame(prompts_data, columns=["prompt", "class1", "class2", "category", "phrase"])
    df.to_csv(csv_name, index=False)
    return csv_name

In [11]:
# for category_list in simple_positional_word_groups:
#     generate_simple_positional_prompts(category_list)

In [12]:
# generate_prompts(words, category="negate_words")

In [13]:
generate_prompts(negative_words, category="negative_words")

Generated 202 prompts
[('a photo of a class1 and not a class2', 'class1', 'class2', 'negative_words', 'and not'), ('a photo of a class1 without a class2', 'class1', 'class2', 'negative_words', 'without'), ('a photo of a giraffe and not a hair drier', 'giraffe', 'hair drier', 'negative_words', 'and not'), ('a photo of a giraffe without a hair drier', 'giraffe', 'hair drier', 'negative_words', 'without'), ('a photo of a broccoli and not a skis', 'broccoli', 'skis', 'negative_words', 'and not')]


'simple_positional_22479673.csv'

In [26]:
action_words = ["sitting", "running", "standing"] # single object actions
objects = ["giraffe", "monkey", "girl", "cat", "dog", "boy", "old man", "old woman", 
           "tiger", "lion", "elephant", "zebra", "bear", "horse", "sheep", "cow", 
           "deer", "fox", "rabbit", "panda", "kangaroo", "koala", "wolf", "leopard", 
           "crocodile", "child", "hippopotamus", "rhinoceros", "buffalo", "camel", "penguin"]
print(len(objects))

31


In [27]:
def generate_single_class_prompts(category_list, category="action_word"):
    """
    generate 100 prompts each of the form 
    "a photo of a {class} {action_verb}"
    """
    prompts_data = []
    for class1 in objects:
        for phrase in category_list:
            prompt = f"a photo of a {class1} {phrase}"
            prompts_data.append((prompt, class1, category, phrase)) 
    print(f"Generated {len(prompts_data)} prompts")

    # save prompts_data to a csv file with unique identifier
    hash_suffix = abs(hash(tuple(category_list))) % (10 ** 8)
    csv_name = f"single_class_{hash_suffix}.csv"
    df = pd.DataFrame(prompts_data, columns=["prompt", "class1", "category", "phrase"])
    df.to_csv(csv_name, index=False)
    return csv_name

In [28]:
generate_single_class_prompts(action_words, category="action_word")

Generated 93 prompts


'single_class_23991210.csv'